# Planner Classes Tutorial

This notebook shows the planning data model used before running RPL.

Core idea:

- A **site** is a physical place with coordinates.
- A **device** is equipment installed at a site.
- An **interface** is the connectable RF endpoint of a device.
- An **antenna** is a reusable catalog object referenced by `antenna_id`.
- RPL sees interfaces as graph nodes.

Interface IDs follow:

```text
site:device:interface
```

Example:

```text
copel:d0:i0
```


In [ ]:
from math import inf

from cisei_lib.planners.planner_classes import (
    AntennaSpec,
    FieldSite,
    RadioInterfacePattern,
    SitePattern,
    RadioInterface,
    Site,
    Device,
    DevicePattern,
    TowerSite,
    get_utm_epsg,
    make_interface_id,
    make_device_id,
)
from cisei_lib.planners.geo_rpl_agnostic import GeoRPL

WORKING_CRS = "EPSG:31982"  # SIRGAS 2000 / UTM zone 22S
WORKING_CRS

## 1. Create One Wi-SUN Field Node

`FieldSite.with_default_interface(...)` is a convenience constructor. It creates:

- one site
- one device
- one interface
- one `antenna_id` reference

The actual antenna definitions live in an external catalog.


In [ ]:
antenna_catalog = {
    "omni_6dbi_7m": AntennaSpec(
        kind="omni",
        gain_dbi=6.25,
        height_m=7,
    ),
    "omni_6dbi_30m": AntennaSpec(
        kind="omni",
        gain_dbi=6.25,
        height_m=30,
    ),
    "sector_0": AntennaSpec(kind="sector", gain_dbi=17, height_m=100, azimuth_deg=0, beamwidth_deg=120, max_links=64),
    "sector_120": AntennaSpec(kind="sector", gain_dbi=17, height_m=100, azimuth_deg=120, beamwidth_deg=120, max_links=64),
    "sector_240": AntennaSpec(kind="sector", gain_dbi=17, height_m=100, azimuth_deg=240, beamwidth_deg=120, max_links=64),
    "directional_14dbi_7m": AntennaSpec(kind="directional", gain_dbi=14, height_m=7),
}

copel = FieldSite.with_default_interface(
    site_id="copel",
    lat=-25.432730835150082,
    lon=-49.339207159685344,
    tech="wisun",
    freq_mhz=900,
    tx_power_dbm=20,
    antenna_id="omni_6dbi_7m",
    fixed=False,
    relay=True,
    rank=inf,
)

copel.resolve_position(WORKING_CRS)

print("site:", copel.site)
print("device:", copel.devices[0])
print("interface:", copel.interfaces[0])
print("antenna:", antenna_catalog[copel.interfaces[0].antenna_id].to_dict())
print("rpl node:", copel.rpl_nodes()[0])


## 2. Use Patterns for Repeated Devices

Patterns avoid repeating the same device/interface configuration for thousands of similar devices.

Pattern keys are local:

```text
d0, i0
```

Instantiated IDs are global:

```text
barigui:d0:i0
```


In [ ]:
wisun_pattern = SitePattern.default_wisun(
    tech="wisun",
    freq_mhz=900,
    tx_power_dbm=20,
    antenna_id="omni_6dbi_7m",
    fixed=False,
    relay=True,
    rank=inf,
)

rows = [
    {"id": "tenis", "lat": -25.425356577135183, "lon": -49.29212222420152},
    {"id": "barigui", "lat": -25.428858965702883, "lon": -49.31129716330312},
    {"lat": -25.427623857466518, "lon": -49.28564299624496},
    {"site_id": "utm_only", "x": 667000.0, "y": 7186000.0},
]

field_nodes = wisun_pattern.instantiate_many(
    rows,
    id_prefix="wisun",
    working_crs=WORKING_CRS,
    resolve=True,
)

for node in field_nodes:
    print(node.node_id, "->", node.rpl_nodes()[0].node_id, node.site.projected_pos)


Rows without `site_id`, `id`, or `name` receive automatic IDs using `id_prefix`.

Projected-only rows are allowed, but `x/y` must already be in `WORKING_CRS`.

In [ ]:
try:
    wisun_pattern.instantiate_many([
        {"id": "duplicated", "lat": -25.1, "lon": -49.1},
        {"id": "duplicated", "lat": -25.2, "lon": -49.2},
    ])
except ValueError as error:
    print(error)

## 3. Build a Tower With Sector Interfaces

A tower may have multiple interfaces at the same site. Each sector interface points to its own antenna catalog entry.


In [ ]:
tower_pattern = SitePattern.tower_sectors(
    antenna_ids=["sector_0", "sector_120", "sector_240"],
    tech="lte",
    freq_mhz=900,
    tx_power_dbm=43,
    fixed=True,
    relay=True,
    rank=0.0,
)

torre = tower_pattern.instantiate(
    "torre",
    lat=-25.423532434217368,
    lon=-49.29399235240344,
    working_crs=WORKING_CRS,
    resolve=True,
)

for interface in torre.interfaces:
    antenna = antenna_catalog[interface.antenna_id]
    print(interface.interface_id, interface.antenna_id, antenna.to_dict())


The older convenience constructor still works and delegates to the pattern mechanism.

In [ ]:
legacy_style_tower = TowerSite.with_sector_interfaces(
    "tower_a",
    lat=-25.40,
    lon=-49.30,
    antenna_ids=["sector_0", "sector_120"],
)

print([interface.interface_id for interface in legacy_style_tower.interfaces])


## 4. Manual Construction

Manual construction is useful when a future antenna planner expands logical interfaces into physical radios/interfaces.

In [ ]:
site = Site(
    site_id="manual_site",
    lat=-25.43,
    lon=-49.30,
)

device = Device(
    device_id=make_device_id("manual_site", 0),
    site_id="manual_site",
)

interface = RadioInterface(
    interface_id=make_interface_id("manual_site", 0, 0),
    device_id=device.device_id,
    site_id="manual_site",
    tech="backhaul",
    freq_mhz=900,
    tx_power_dbm=27,
    antenna_id="directional_14dbi_7m",
    fixed=False,
    relay=True,
    rank=inf,
)

site.resolve(WORKING_CRS)
interface.to_rpl_node(site, device)


## 5. Convert Interfaces to RPL Nodes

`GeoRPL` receives only simplified `RPLNode` objects and weighted feasible edges. It does not know about sectors, antenna gain, technology, or radio inventory.

In [ ]:
gateway_pattern = SitePattern.default_wisun(
    tech="wisun",
    freq_mhz=900,
    tx_power_dbm=20,
    antenna_id="omni_6dbi_30m",
    fixed=True,
    relay=True,
    rank=0.0,
)

gateway = gateway_pattern.instantiate(
    "gateway",
    lat=-25.43017531546412,
    lon=-49.266232140399694,
    working_crs=WORKING_CRS,
    resolve=True,
)

planning_nodes = [gateway, *field_nodes]
rpl_nodes = []

for node in planning_nodes:
    rpl_nodes.extend(node.rpl_nodes())

for node in rpl_nodes:
    print(
        f"{node.node_id:18s}",
        "fixed=", node.fixed,
        "relay=", node.relay,
        "pos_utm", node.pos_utm,
        "site=", node.extra["site_id"],
        "device=", node.extra["device_id"],
        "tech=", node.extra.get("tech"),
        "freq_mhz=", node.extra.get("freq_mhz"),
        "tx_power_dbm=", node.extra.get("tx_power_dbm"),
        "antenna_id=", node.extra.get("antenna_id"),
    )


## 6. Run RPL With Fake Edge Metrics

In production, a graph-preparation class will decide which links are feasible and compute metrics. Here we use simple fake costs to show the data flow.

In [ ]:
planner = GeoRPL()
planner.set_nodes(rpl_nodes)

gateway_id = gateway.rpl_nodes()[0].node_id
field_ids = [node.rpl_nodes()[0].node_id for node in field_nodes]

edges = {}

for index, node_id in enumerate(field_ids, start=1):
    edges[(gateway_id, node_id)] = float(index)

for left, right in zip(field_ids, field_ids[1:]):
    edges[(left, right)] = 1.0

planner.set_edge_metrics(edges)
planner.run_RPL()

for node_id, attrs in planner.G_res.nodes(data=True):
    print(
        f"{node_id:18s}",
        "parent=", attrs.get("parent"),
        "rank=", round(attrs["rank"], 3),
    )

In [ ]:
planner.show_network(G=planner.G_res)